In [1]:
from autoplex.auto.GenMLFF.flows import GenMLFlow
from jobflow_remote import submit_flow, set_run_config

/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/nequip/__init__.py:20: UserWarning: !! PyTorch version 2.2.1+cu121 found. Upstream issues in PyTorch versions 1.13.* and 2.* have been seen to cause unusual performance degredations on some CUDA systems that become worse over time; see https://github.com/mir-group/nequip/discussions/311. The best tested PyTorch version to use with CUDA devices is 1.11; while using other versions if you observe this problem, an unexpected lack of this problem, or other strange behavior, please post in the linked GitHub issue.
  warnings.warn(


In [2]:
#Define resources
parallel_cpu_resources = {
    "account": "EUHPC_A04_113",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 32,
    "cpus_per_task": 1,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_cpu_resources = {
    "account": "EUHPC_A04_113",
    "partition": "lrd_all_serial",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 8,
    "gres": "gpu:0",
    "mem": "30000",
    "job_name": "Sampling",
    "qerr_path": "Sampling.err",
    "qout_path": "Sampling.out",
}

serial_gpu_resources = {
    "account": "EUHPC_A04_113", 
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 8,
    "gres": "gpu:1",
    "mem": "120000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
    }

In [3]:
#Instantiate the GenMLFlow
GenML = GenMLFlow(input_fname="/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/Test-workflow/rss-mlip/GenML_config.yaml")

In [4]:
GenML = set_run_config(
    GenML, name_filter="RSS", worker="mlff_relax_local", exec_config="rss_config", resources=serial_cpu_resources
)

GenML = set_run_config(
    GenML, name_filter="MatterGenGenerate", exec_config="matgen_config", worker="MatterGen", resources=serial_gpu_resources
)

GenML = set_run_config(
    GenML, name_filter="evaluate_mlip_ensemble", worker="mlff_mace", resources=serial_gpu_resources
)

GenML = set_run_config(
    GenML, name_filter="atomic_configuration_sampling", worker="mlff_relax_local", resources=serial_cpu_resources
)

GenML = set_run_config(
    GenML, name_filter="run_qe_worker", exec_config="qe_config", worker="QuantumEspresso", resources=serial_gpu_resources
)

GenML = set_run_config(
    GenML, name_filter="training_mlip", worker="mlff_relax_local", exec_config="mace_config", resources=serial_gpu_resources
)

In [5]:
# Append RSSautoplex-flow to jf jobs
submit_flow(
    GenML, worker="local_worker",
    resources={}, 
    project="GenMLFF",
)

2025-07-03 16:59:19,899 - INFO - Added flow (30362bb2-31a7-43db-8593-d37e7cacd749) with jobs: ('a86b0d31-af09-4064-b9df-442ae1af47e2',)


['1']